# Random number generation

This notebook is a gentle introduction to pseudo-random number generation. It presents some of the underlying concepts, and the implementations favour readability over speed. Efficient implementations are available online, and should be preferred for serious purposes.

## Linear congruential generators

We first consider general LCGs, of the form
$$
x_{k+1} = a x_k + c \mod m.
$$
The output is $u_k = x_k / m$. Note that the recurrence is evaluated in machine integers: $a x_k + c$ must remain representable, i.e. $a(m-1) + c$ has to stay below `typemax(Int64)` when the state is an `Int64`.

Adapted from https://rosettacode.org/wiki/Linear_congruential_generator

In [ ]:
function getlcg(seed::Integer, a::Integer, c::Integer, m::Integer)
    state = Ref{typeof(seed)}(seed)
    invm = 1.0/m
    return function lcgrand()
        state[] = mod(a * state[] + c, m)
        return state[]*invm  # in [0, 1), and in (0, 1) when c = 0 and m is prime
    end
end

Standard minimal generator (Lehmer, popularized by Park and Miller): $a = 7^5 = 16807$, $c = 0$, $m = 2^{31} - 1$.

In [ ]:
stdmin = getlcg(1234, 16807, 0, 2^31-1)

In [ ]:
n = 10000

sample = zeros(n)

for i = 1:n
    sample[i] = stdmin()
end

In [ ]:
using Plots

In [ ]:
scatter(sample[[2*i+1 for i = 0:(Int)(n/2)-1]], sample[[2*i for i = 1:(Int)(n/2)]], label="", fmt = :png)

In [ ]:
n = 100000

sample = zeros(n)

for i = 1:n
    sample[i] = stdmin()
end

scatter(sample[[2*i+1 for i = 0:(Int)(n/2)-1]], sample[[2*i for i = 1:(Int)(n/2)]], label="", fmt = :png)

## RandomDataStreams

A Julia package providing a unified framework for independent streams and substreams.

Two versions matter here:

- the **registered** version (v0.1.0), which covers MRG32k3a and the xoshiro family;
- the **`Philox` branch** (v0.2.0), which adds the counter-based generators of the course
  (Philox, Threefry) and the PCG family. It is expected to be registered in late autumn 2026;
  until then the branch is the only way to get them.

We use the branch below.

### Installing a given branch

`Pkg.add` takes a `rev` keyword, which accepts a branch name, a tag or a commit SHA.

| what you want | how |
| --- | --- |
| the registered version | `Pkg.add("RandomDataStreams")` |
| a branch of a registered package | `Pkg.add(name = "RandomDataStreams", rev = "Philox")` |
| any repository, registered or not | `Pkg.add(url = "https://github.com/JLChartrand/RandomDataStreams.jl.git", rev = "Philox")` |
| one precise commit | `Pkg.add(name = "RandomDataStreams", rev = "5b3b7e2")` |
| back to the registered version | `Pkg.free("RandomDataStreams")` |

In the package REPL, the short form `] add RandomDataStreams#Philox` does the same thing.
Beware that this `#` shorthand belongs to the REPL only: passed to the function API,
`Pkg.add("RandomDataStreams#Philox")` fails with `is not a valid package name`.

`Pkg.status()` shows which one is currently active --- a package taken from a branch is
displayed with its repository URL and revision.

In [ ]:
import Pkg

# libgit2 may ask for SSH credentials and abort in a non-interactive session
# (a notebook launched without a terminal, a CI job); delegating to the system
# git avoids it.
ENV["JULIA_PKG_USE_CLI_GIT"] = "true"

Pkg.add(name = "RandomDataStreams", rev = "Philox")   # v0.2.0
Pkg.add("RNGTest")

In [ ]:
Pkg.status("RandomDataStreams")

In [ ]:
using RandomDataStreams

### MRG32K3a

In [ ]:
mrg_gen1 = MRG32k3aGen([1,2,3,4,5,6])

In [ ]:
typeof(mrg_gen1)

In [ ]:
mrg_gen1 = MRG32k3aGen()

The `show` function prints the seed that will be used for the *next* stream produced by the generator.

In [ ]:
show(mrg_gen1)

In [ ]:
mrg_1 = next_stream!(mrg_gen1)

In [ ]:
typeof(mrg_1)

In [ ]:
mrg_1 = next_stream!(mrg_gen1)

In [ ]:
stream1a = [rand(mrg_1) for i in 1:10]
reset_substream!(mrg_1)
stream1b = [rand(mrg_1) for i in 1:10]
stream1a == stream1b

In [ ]:
show(mrg_gen1)

In [ ]:
next_substream!(mrg_1)

In [ ]:
stream2a = [rand(mrg_1) for i in 1:10]
reset_stream!(mrg_1)

In [ ]:
next_substream!(mrg_1)
stream2b = [rand(mrg_1) for i in 1:10]
stream2a == stream2b

In [ ]:
reset_stream!(mrg_1)
mrg_2 = next_stream!(mrg_gen1)

### Xoshiro256++
Similar operations can be performed with the `xoshiro256++` generator. Julia's own default
random number generator (`Random.TaskLocalRNG`) belongs to the same Xoshiro family, but the
`Xoshiro256ppGen` used below is the stream-splitting variant provided by `RandomDataStreams`;
the two are unrelated objects.

In [ ]:
xosh_gen = Xoshiro256ppGen([UInt64(1), UInt64(2), UInt64(3), UInt64(4)])

In [ ]:
typeof(xosh_gen)

In [ ]:
show(xosh_gen)

In [ ]:
xosh_1 = next_stream!(xosh_gen)

In [ ]:
typeof(xosh_1)

In [ ]:
xosh_1 = next_stream!(xosh_gen)

In [ ]:
stream1a_xosh = [rand(xosh_1) for i in 1:10]
reset_substream!(xosh_1)
stream1b_xosh = [rand(xosh_1) for i in 1:10]
stream1a_xosh == stream1b_xosh

In [ ]:
show(xosh_gen)

In [ ]:
next_substream!(xosh_1)

In [ ]:
stream2a_xosh = [rand(xosh_1) for i in 1:10]
reset_stream!(xosh_1)

In [ ]:
next_substream!(xosh_1)
stream2b_xosh = [rand(xosh_1) for i in 1:10]
stream2a_xosh == stream2b_xosh

In [ ]:
reset_stream!(xosh_1)
xosh_2 = next_stream!(xosh_gen)

### Counter-based generators

The `Philox` branch adds the counter-based generators presented in the slides, plus the PCG
family. They expose exactly the same interface as the generators above --- a generator object
produces independent streams, each stream carries substreams --- so nothing in the way we use
them changes.

In [ ]:
for (name, G) in (("Philox4x32-10", PhiloxGen),     ("Philox4x64-10", Philox4x64Gen),
                  ("Threefry4x32",  Threefry4x32Gen), ("Threefry4x64",  Threefry4x64Gen),
                  ("PCG64",         PCG64Gen),        ("PCG64-DXSM",    PCG64DXSMGen))
    gen = G(1234)
    s1, s2 = next_stream!(gen), next_stream!(gen)
    first1 = [rand(s1) for _ in 1:3]
    first2 = [rand(s2) for _ in 1:3]
    reset_stream!(s1)
    println(rpad(name, 15),
            " reproducible: ", rpad(first1 == [rand(s1) for _ in 1:3], 6),
            " independent of stream 2: ", first1 != first2)
end

The `show` methods report the key (which identifies the stream) and the counter (the
position inside it) --- the two halves of the CBRNG state described in the slides.

In [ ]:
phlx_gen = PhiloxGen(1234)
show(phlx_gen)
println()

phlx_1 = next_stream!(phlx_gen)
show(phlx_1)

Substreams behave as for MRG32k3a, except that they are obtained by moving the high half
of the counter rather than by a precomputed jump matrix.

In [ ]:
sub0 = [rand(phlx_1) for _ in 1:5]
next_substream!(phlx_1)
sub1 = [rand(phlx_1) for _ in 1:5]

reset_stream!(phlx_1)
next_substream!(phlx_1)
println("substream 1 reproducible : ", sub1 == [rand(phlx_1) for _ in 1:5])
println("distinct from substream 0: ", sub0 != sub1)

#### Jumping ahead

This is what counter-based generators are for. Because the state *is* the position, moving
$n$ draws forward is an addition on the counter --- it does not cost more than moving one draw
forward, and it lands exactly where the sequential draws would have.

In [ ]:
# Reference: one million draws, keeping the last three.
gen_seq = PhiloxGen(2024)
r_seq   = next_stream!(gen_seq)
sequential = [rand(r_seq) for _ in 1:1_000_000]

# Same stream, jumping straight to the same position.
gen_jmp = PhiloxGen(2024)
r_jmp   = next_stream!(gen_jmp)
advance_state!(r_jmp, 0, 999_997)          # skip 999_997 draws; the next one is #999_998
jumped = [rand(r_jmp) for _ in 1:3]

println("last three, drawn one by one : ", round.(sequential[end-2:end], digits = 6))
println("last three, after the jump   : ", round.(jumped, digits = 6))
println("identical                    : ", sequential[end-2:end] == jumped)

In [ ]:
using BenchmarkTools

t_draw = @belapsed (g = PhiloxGen(2024); r = next_stream!(g);
                    for _ in 1:1_000_000; rand(r); end) seconds=2
t_jump = @belapsed (g = PhiloxGen(2024); r = next_stream!(g);
                    advance_state!(r, 0, 999_997))                seconds=2

println("drawing 10^6 values : ", round(t_draw * 1e3, digits = 2), " ms")
println("jumping over them   : ", round(t_jump * 1e6, digits = 2), " µs")
println("ratio               : ", round(Int, t_draw / t_jump), "×")

That property is what makes CBRNGs natural for parallel simulation: a worker can be handed
a position rather than a state, and no worker has to wait for another to advance the sequence.
It is also why `advance_state!` is cheap here, whereas for MRG32k3a the same operation requires
precomputed jump matrices.

## Nonuniform distributions

For continuous random variables, the inversion technique amounts to computing the quantile $F^{-1}(U)$ associated with the realization of a uniform random variable $U(0,1)$. We will use the `Distributions` package.

In [ ]:
using Distributions

In [ ]:
# N is declared const: see the remark before the benchmarks below.
const N = Normal()
z975 = quantile(N, 0.975)

Normally distributed number generation:

In [ ]:
quantile(N, rand())

We can measure the required generation time with the package `BenchmarkTools`.

Two precautions matter here. First, `N` was declared `const`: benchmarking a function that
reads a *non-constant* global would mostly measure the cost of Julia's type instability, not
the cost of the inversion itself. Second, we time `f()` rather than `X, Y = f()`, so that the
assignment to global variables is not included in the measurement.

In [ ]:
using BenchmarkTools

In [ ]:
function InvertNormal()
    U = rand(Float64, 2)
    return quantile(N, U[1]), quantile(N, U[2])
end

In [ ]:
@btime InvertNormal()

In [ ]:
function InvertNormal2()
    return quantile(N, rand(Float64)), quantile(N, rand(Float64))
end

In [ ]:
@btime InvertNormal2()

In [ ]:
function BoxMuller()
    U = rand(Float64, 2)

    # rand() returns values in [0, 1); 1 - U[1] lies in (0, 1] and keeps the log finite.
    R = sqrt(-2*log(1 - U[1]))
    θ = 2*π*U[2]

    X = R*cos(θ)
    Y = R*sin(θ)

    return X, Y
end

In [ ]:
@btime BoxMuller()

In [ ]:
randn()

In [ ]:
@btime randn()

Inversion and Box-Muller have comparable costs, and both are far slower than `randn()`,
which relies on the *ziggurat* algorithm. Inversion nevertheless remains attractive in
simulation: it is monotone in $U$ and consumes exactly one uniform per variate, which makes
it compatible with common random numbers and with quasi-Monte Carlo. The next section
explains where the gap comes from, and what it costs.

## Why is `randn()` so fast?

Julia generates normal variates with the *ziggurat* method (Marsaglia and Tsang, 2000);
the implementation is in `Random/src/normal.jl`. The density is covered by 256 horizontal
layers of equal area, precomputed once and stored in three tables of 256 entries
(`ki`, `wi`, `fi`). The whole fast path is then:

```julia
r    = rand(rng, UInt52())              # a single draw
rabs = Int64(r >> 1)                    # one bit is kept for the sign
idx  = rabs & 0xFF                      # 8 bits select the layer
x    = ifelse(r % Bool, -rabs, rabs) * wi[idx+1]
rabs < ki[idx+1] && return x            # accept, and we are done
return randn_unlikely(rng, idx, rabs, x)
```

One draw, one mask, one table lookup, one multiplication, one comparison --- and **no
transcendental function at all**. Let us check how often that path is taken.

In [ ]:
using Random

"""
    ziggurat_fast_path_rate(niter)

Fraction of the draws accepted on the ziggurat's first try, i.e. without ever
entering `randn_unlikely`.
"""
function ziggurat_fast_path_rate(niter = 10_000_000; rng = Xoshiro(1))
    ki = Random.ki
    fast = 0
    @inbounds for _ in 1:niter
        r = rand(rng, Random.UInt52())
        rabs = Int64(r >> 1)
        idx = rabs & 0xFF
        fast += (rabs < ki[idx+1])
    end
    return fast / niter
end

println(round(100 * ziggurat_fast_path_rate(), digits = 2), " % of the draws accept immediately")

To compare the methods fairly, we time them inside a tight loop, so that the measurement
is not dominated by the cost of an isolated function call (this is why the numbers below are
smaller than the `@btime` figures above --- both are correct, they simply measure different
things).

In [ ]:
const NDRAW    = 200_000
const UNIFORMS = rand(NDRAW)

sum_rand()  = (s = 0.0; for _ in 1:NDRAW; s += rand();  end; s)
sum_randn() = (s = 0.0; for _ in 1:NDRAW; s += randn(); end; s)

function sum_inversion(u)
    s = 0.0
    @inbounds for x in u
        s += quantile(N, x)
    end
    return s
end

function sum_boxmuller(u)
    s = 0.0
    @inbounds for i in 1:2:length(u)
        R = sqrt(-2*log(1 - u[i]))
        θ = 2*π*u[i+1]
        s += R*cos(θ) + R*sin(θ)
    end
    return s
end

per_value(t) = string(round(t * 1e9 / NDRAW, digits = 2), " ns / value")

println("rand()          : ", per_value(@belapsed sum_rand()          seconds=1))
println("randn()         : ", per_value(@belapsed sum_randn()         seconds=1))
println("quantile(N, u)  : ", per_value(@belapsed sum_inversion($UNIFORMS)  seconds=1))
println("Box-Muller      : ", per_value(@belapsed sum_boxmuller($UNIFORMS)  seconds=1))

`randn()` costs barely more than the uniform draw it consumes: the normal transformation
itself is worth about one nanosecond. The other two methods must evaluate transcendental
functions --- $\Phi^{-1}$ for inversion (a rational approximation with branches per region),
and `log`, `sqrt`, `cos`, `sin` for Box-Muller. The ziggurat performs that work **once,
offline**: it is baked into the tables, which occupy some 6 KB and stay in L1 cache.

Two further implementation details matter:

- **bit reuse** --- a single 52-bit draw supplies the sign (1 bit), the layer index (8 bits)
  and the position inside the layer (the rest);
- **branch layout** --- the rare cases (the exponential tail beyond $r = 3.654$, and the
  wedge between a rectangle and the density) live in a separate `@noinline` function, so the
  hot path stays small and the branch predictor is almost never wrong.

### The price: a rejection method

The ziggurat accepts or rejects, so the number of uniforms it consumes is *random*. We can
measure it by re-implementing `randn` faithfully with a counter, and checking that the
re-implementation reproduces `Random.randn` exactly.

In [ ]:
"""
    randn_counted(rng, cnt)

Re-implementation of `Random.randn` that increments `cnt` for every uniform drawn.
"""
function randn_counted(rng, cnt::Ref{Int})
    ki, wi, fi = Random.ki, Random.wi, Random.fi
    r_tail, inv_r = Random.ziggurat_nor_r, Random.ziggurat_nor_inv_r
    @inbounds begin
        cnt[] += 1
        r = rand(rng, Random.UInt52())
        rabs = Int64(r >> 1)
        idx = rabs & 0xFF
        x = ifelse(r % Bool, -rabs, rabs) * wi[idx+1]
        rabs < ki[idx+1] && return x                     # fast path
        if idx == 0                                      # exponential tail
            while true
                cnt[] += 2
                xx = -inv_r*log1p(-rand(rng))
                yy = -log1p(-rand(rng))
                yy + yy > xx*xx && return (rabs >> 8) % Bool ? -r_tail - xx : r_tail + xx
            end
        elseif (cnt[] += 1; (fi[idx] - fi[idx+1])*rand(rng) + fi[idx+1] < exp(-0.5*x*x))
            return x                                     # wedge below the density
        else
            return randn_counted(rng, cnt)               # start over
        end
    end
end

reference = (rng = Xoshiro(2024); [randn(rng) for _ in 1:1000])
replica   = (rng = Xoshiro(2024); c = Ref(0); [randn_counted(rng, c) for _ in 1:1000])
println("reproduces Random.randn exactly : ", reference == replica)

rng, counter, nrep = Xoshiro(7), Ref(0), 5_000_000
for _ in 1:nrep
    randn_counted(rng, counter)
end
println("randn()   : ", round(counter[] / nrep, digits = 4), " uniform draws per variate")
println("inversion : 1 uniform draw per variate, always")

### Limitations

The speed is bought with exactly the property that made inversion attractive.

1. **No fixed map from a uniform to a variate.** The number of uniforms consumed is
   random --- just above 1 on average, but 1, 2, 3, ... on any given call. Common random
   numbers, antithetic variates and quasi-Monte Carlo all rely on that map, so none of them
   can be used with `randn()`. This is the same objection as for acceptance-rejection in
   general.

2. **Not invertible, not monotone.** One cannot ask which $u$ produced a given $x$. That
   rules out stratified sampling and inversion-based importance sampling, and makes a
   simulation harder to debug or to reproduce partially.

3. **The tables are distribution-specific.** Julia ships `ki, wi, fi` for the normal and
   `ke, we, fe` for the exponential --- 256 entries each. The method needs a density that is
   monotone decreasing (or symmetric unimodal) with a tractable tail, and every new
   distribution requires its own precomputed tables. Inversion applies to *any* $F$,
   including empirical and discrete ones, and to distributions whose parameters change from
   one call to the next.

4. **Bit reuse makes the output sensitive to the low-order bits of the generator.** The layer
   index is `rabs & 0xFF`, i.e. the low 8 bits, and those same bits also contribute to the
   position inside the layer: the two are not independent. Ziggurat implementations have
   historically been a source of subtle statistical defects for that reason (Doornik, 2005).
   Julia draws 52 bits from `xoshiro256++`, whose low-order bits are sound, which is what
   makes the reuse safe here --- it would not be with a generator whose low bits are weak,
   such as an LCG with $m = 2^k$.

5. **The stream depends on the algorithm, not only on the seed.** Two methods fed by the
   same generator and the same seed produce different sequences, so a simulation cannot be
   reproduced across libraries that do not share the generation algorithm. The last point is
   easy to see:

In [ ]:
r1 = Xoshiro(11); println("randn(rng)             : ", round.([randn(r1)          for _ in 1:5], digits = 4))
r2 = Xoshiro(11); println("rand(rng, Normal())    : ", round.([rand(r2, Normal())  for _ in 1:5], digits = 4))
r3 = Xoshiro(11); println("quantile(N, rand(rng)) : ", round.([quantile(N, rand(r3)) for _ in 1:5], digits = 4))

`Distributions.rand(rng, Normal())` simply delegates to `randn`, hence the two identical
first lines; inversion, fed by the very same uniform stream, produces something else
entirely. Reproducing a simulation therefore requires fixing the generation method, not only
the generator and its seed.

## Testing generators with TestU01
The `RNGTest` package gives access to the TestU01 batteries. Two entry points exist for
SmallCrush:

- `RNGTest.smallcrushTestU01(f)` runs the battery and prints TestU01's own report;
- `RNGTest.smallcrushJulia(f)` runs the same ten tests but *returns* the p-values without
  printing anything.

We use the second one and label the results ourselves. In both cases `f` must be a
zero-argument function returning a `Float64` in $[0, 1)$.

In [ ]:
using RNGTest

In [ ]:
const SMALLCRUSH_TESTS = ["BirthdaySpacings", "Collision", "Gap", "SimpPoker",
                          "CouponCollector", "MaxOft", "WeightDistrib",
                          "MatrixRank", "HammingIndep", "RandomWalk1"]

"""
    report_smallcrush(f)

Run the SmallCrush battery on the zero-argument generator `f` and print the p-values,
flagging those outside `[1e-3, 1 - 1e-3]`. Some tests return several p-values.
"""
function report_smallcrush(f)
    pvalues = RNGTest.smallcrushJulia(f)
    nsuspect = 0
    for (name, p) in zip(SMALLCRUSH_TESTS, pvalues)
        ps = p isa Tuple ? collect(p) : [p]
        flags = [(x < 1e-3 || x > 1 - 1e-3) ? "  <-- suspect" : "" for x in ps]
        println(rpad(name, 18), join(string.(round.(ps, sigdigits = 4), flags), ", "))
        nsuspect += count(!isempty, flags)
    end
    ntotal = sum(p isa Tuple ? length(p) : 1 for p in pvalues)
    println("\n", nsuspect, " suspect p-value(s) out of ", ntotal)
    return pvalues
end

We first test our `stdmin` generator. We rebuild it so that the battery starts from
the documented seed rather than from the state left by the earlier cells.

In [ ]:
lcg = getlcg(1234, 16807, 0, 2^31-1)
report_smallcrush(lcg);

The minimal standard generator fails `BirthdaySpacings`, `Collision` and `MaxOft`
with p-values numerically equal to 0. This is expected: its period, $2^{31} - 2 \approx 2.1 \times 10^9$,
is too short for a battery that consumes several million values per test, and its points lie on
a coarse lattice (the pattern already visible in the scatter plots above).

We can also test a stream from the `RandomDataStreams` library. Since
`report_smallcrush` expects a zero-argument function returning a `Float64`, we wrap the
stream's `rand` call in an anonymous function.

In [ ]:
mrg_test_gen = MRG32k3aGen()
mrg_test_stream = next_stream!(mrg_test_gen)

report_smallcrush(() -> rand(mrg_test_stream));

MRG32k3a passes the whole battery, as expected from a generator designed for
simulation: period close to $2^{191}$, good lattice structure, and support for independent
streams and substreams.

Finally, the counter-based generator from the `Philox` branch. Its output function is a
non-cryptographic block cipher, so the statistical quality does not come from a long
recurrence but from the mixing performed by the ten rounds.

In [ ]:
phlx_test_gen    = PhiloxGen(1234)
phlx_test_stream = next_stream!(phlx_test_gen)

report_smallcrush(() -> rand(phlx_test_stream));

Philox4x32-10 passes the battery as well. The three runs together make the point of the
chapter: a generator that fails is not merely *old*, it is structurally deficient --- the
minimal standard generator fails because its period is short and its points lie on a coarse
lattice, whereas both MRG32k3a and Philox were designed against precisely those tests.